In [2]:
import scipy.stats as sps
import numpy as np

In [6]:
f0 = sps.norm(-3,2)
f1 = sps.norm(3,2)
prior = 0.25

N = 1000
x = np.concatenate((f1.rvs(int(np.round(N*prior))),
                    f0.rvs(int(np.round(N * (1 - prior))))))

In [7]:
def mlls(x, pathogenic_density, benign_density):
    prior_estimate = 0.5
    converged = False
    tolerance = 1e-6
    em_steps = 0
    max_em_steps = 10000
    while not converged:
        em_steps += 1
        posteriors = 1 / (
            1
            + (1 - prior_estimate)
            / prior_estimate
            * benign_density # type: ignore
            / pathogenic_density
        )
        new_prior = np.nanmean(posteriors)
        relative_change = np.abs(new_prior - prior_estimate)/prior_estimate
        if relative_change < tolerance or np.isnan(new_prior):
            converged = True
        prior_estimate = new_prior
        if prior_estimate < 0 or prior_estimate > 1:
            raise ValueError(f"Invalid prior estimate obtained, {prior_estimate}")
        if em_steps >= max_em_steps:
            print(f"EM prior estimate algorithm failed to converge after {max_em_steps:,d} iterations. ")
            break
    return prior_estimate

In [8]:
mlls(x, f1.pdf(x), f0.pdf(x))

np.float64(0.26110552838902995)